# Silver Layer: FIFA World Cup Match Data

This notebook implements the **silver (cleansed & enriched)** layer for FIFA World Cup match data.

**Purpose:**
- Read validated data from the bronze Delta table
- Apply business logic: derive match winner (including penalty shootout resolution)
- Standardize all string fields, handle nulls, deduplicate
- Validate silver-level data quality (single-pass, Spark advisory optimized)
- Persist as Delta table for downstream gold layer / ML consumption

**Source:** `fifa_worldcup.match_prediction_dev.matches_bronze` 
**Target:** `fifa_worldcup.match_prediction_dev.matches_silver` (Unity Catalog)

**SLA:** On-demand | Max staleness: configurable via `freshness_hours` widget (default: 168h)

---

### Transformation Rules
| Derived Column | Logic |
| --- | --- |
| `winner` | If penalty shootout: team with more penalties wins. Otherwise: team with more goals wins. Tie = "Draw" |
| `match_outcome` | `home_win` / `away_win` / `draw` (based on regular-time goals) |
| `goal_difference` | `abs(home_goals - away_goals)` |
| `decided_by_penalties` | `True` if `penalty_shootout == 1` |
| `total_goals` | `home_goals + away_goals` |

---

### Data Contract (Output Schema Guarantee)
The silver table guarantees the following for gold layer / ML consumers:
- `match_id` is unique (deduplicated)
- No nulls in: match_id, match_date, home_team, away_team, home_goals, away_goals, winner
- `match_date` never exceeds current date
- Goal values are non-negative and ≤ 20 per team
- All string fields are trimmed of leading/trailing whitespace
- Winner logic correctly resolves penalty shootouts
- Row preservation ≥ 95% of bronze input

### Data Product Metadata
- **TBLPROPERTIES:** layer, pipeline, source_table, quality_tier, owner, refresh_mode, downstream_tables, transformations
- **COMMENT ON TABLE** for UC discoverability
- **COMMENT ON COLUMN** for 15 key columns (match_id, teams, goals, outcomes, metadata)
- **Data Contract Enforcement** cell validates 6 contract assertions before exit
- **Data Freshness SLA** cell validates bronze source staleness

---

### Orchestration
**Job:** [FIFA Match Prediction - Full Pipeline](/#job/146142328218872) 
**Schedule:** Manual (on-demand) 
**DAG:** `bronze_ingestion` → `silver_transformation` → `gold_analysis` 
**Cluster:** Amar Rathour's Cluster (`0511-052237-lgol1ybz`)

This notebook runs as the second task after bronze completes. Gold analysis depends on this task.

In [0]:
# No external dependencies required for silver layer.
# Silver reads from registered Delta table — no file-system helpers needed.

In [0]:
from datetime import date, datetime, timedelta

# --- Configuration (parameterized for multi-environment support) ---
dbutils.widgets.text("environment", "dev", "Environment (dev/staging/prod)")
dbutils.widgets.text("catalog", "fifa_worldcup", "Unity Catalog Name")
dbutils.widgets.dropdown("refresh_mode", "full", ["full", "incremental"], "Refresh Mode")
dbutils.widgets.text("freshness_hours", "168", "Max staleness (hours)")

ENV = dbutils.widgets.get("environment")
CATALOG = dbutils.widgets.get("catalog")
REFRESH_MODE = dbutils.widgets.get("refresh_mode")
FRESHNESS_HOURS = int(dbutils.widgets.get("freshness_hours"))
SCHEMA_NAME = f"match_prediction_{ENV}"

# --- Source & Target Table Names (Unity Catalog ready) ---
BRONZE_TABLE = f"{CATALOG}.{SCHEMA_NAME}.matches_bronze"
SILVER_TABLE_NAME = "matches_silver"
SILVER_FULL_TABLE = f"{CATALOG}.{SCHEMA_NAME}.{SILVER_TABLE_NAME}"

# --- Constants ---
TODAY = date.today()
MAX_GOALS_PER_TEAM = 20  # Domain constraint: any value above this is likely data corruption

print(f"Environment: {ENV}")
print(f"Catalog: {CATALOG}")
print(f"Refresh mode: {REFRESH_MODE}")
print(f"Max staleness: {FRESHNESS_HOURS} hours")
print(f"Source (bronze): {BRONZE_TABLE}")
print(f"Target (silver): {SILVER_FULL_TABLE}")
print(f"Run date: {TODAY}")

In [0]:
from pyspark.sql.functions import max as spark_max

# ============================================================
# DATA FRESHNESS SLA: Validate bronze source is fresh enough
# Checks _ingestion_timestamp to ensure bronze has run within the
# configured staleness window before processing silver.
# ============================================================

bronze_freshness = (
    spark.table(BRONZE_TABLE)
    .agg(spark_max("_ingestion_timestamp").alias("latest_ingestion_ts"))
    .collect()[0]["latest_ingestion_ts"]
)

freshness_threshold = datetime.now() - timedelta(hours=FRESHNESS_HOURS)

if bronze_freshness is None:
    print("\u26a0\ufe0f WARNING: Bronze table has no _ingestion_timestamp timestamps.")
    print("  Proceeding anyway (first run scenario).")
elif bronze_freshness < freshness_threshold:
    staleness_hours = (datetime.now() - bronze_freshness).total_seconds() / 3600
    error_msg = (
        f"\u274c DATA FRESHNESS SLA VIOLATED\n"
        f"  Bronze last ingested: {bronze_freshness}\n"
        f"  Staleness: {staleness_hours:.1f} hours (max allowed: {FRESHNESS_HOURS})\n"
        f"  Action: Re-run the bronze pipeline before processing silver."
    )
    if ENV == "prod":
        raise RuntimeError(error_msg)
    else:
        print(f"\u26a0\ufe0f {error_msg}")
        print("  (Non-prod: proceeding with stale data)")
else:
    staleness_hours = (datetime.now() - bronze_freshness).total_seconds() / 3600
    print(f"\u2705 Data Freshness SLA: PASS")
    print(f"  Bronze last ingested: {bronze_freshness}")
    print(f"  Staleness: {staleness_hours:.1f} hours (threshold: {FRESHNESS_HOURS}h)")

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, when, lit, trim, lower, coalesce, abs as spark_abs,
    current_timestamp
)

# --- Read from Bronze Delta Table (registered in previous layer) ---
# Optimization: Avoid .count() here — it triggers a full scan just for logging.
# The actual row count is captured in the single-pass validation (Cell 5).
matches_bronze_df = spark.table(BRONZE_TABLE)
print(f"Reading from bronze table: {BRONZE_TABLE}")

# --- Silver Transformations ---
# 1. Select relevant columns and standardize naming
# 2. Derive winner, match outcome, goal difference
# 3. Standardize string fields (trim whitespace)
# 4. Handle nulls with sensible defaults

matches_silver_df = (
    matches_bronze_df
    .select(
        "match_id",
        "tournament_name",
        "match_date",
        "stage_name",
        "stadium_name",
        "city_name",
        "country_name",
        col("home_team_name").alias("home_team"),
        col("home_team_code").alias("home_team_code"),
        col("away_team_name").alias("away_team"),
        col("away_team_code").alias("away_team_code"),
        col("home_team_score").alias("home_goals"),
        col("away_team_score").alias("away_goals"),
        "extra_time",
        "penalty_shootout",
        col("home_team_score_penalties").alias("home_penalties"),
        col("away_team_score_penalties").alias("away_penalties"),
        "_ingestion_timestamp",
        "_batch_id",
        "_row_hash",
    )
    # Derive winner (handles penalty shootout correctly)
    .withColumn(
        "winner",
        when(
            (col("penalty_shootout") == 1) & (col("home_penalties") > col("away_penalties")),
            col("home_team")
        )
        .when(
            (col("penalty_shootout") == 1) & (col("away_penalties") > col("home_penalties")),
            col("away_team")
        )
        .when(col("home_goals") > col("away_goals"), col("home_team"))
        .when(col("away_goals") > col("home_goals"), col("away_team"))
        .otherwise(lit("Draw"))
    )
    # Derive match outcome category
    .withColumn(
        "match_outcome",
        when(col("home_goals") > col("away_goals"), lit("home_win"))
        .when(col("away_goals") > col("home_goals"), lit("away_win"))
        .otherwise(lit("draw"))
    )
    # Compute goal difference
    .withColumn("goal_difference", spark_abs(col("home_goals") - col("away_goals")))
    # Standardize: trim whitespace on ALL string fields
    .withColumn("home_team", trim(col("home_team")))
    .withColumn("away_team", trim(col("away_team")))
    .withColumn("tournament_name", trim(col("tournament_name")))
    .withColumn("stage_name", trim(col("stage_name")))
    .withColumn("stadium_name", trim(col("stadium_name")))
    .withColumn("city_name", trim(col("city_name")))
    .withColumn("country_name", trim(col("country_name")))
    # Handle nulls: default penalties to 0
    .withColumn("home_penalties", coalesce(col("home_penalties"), lit(0)))
    .withColumn("away_penalties", coalesce(col("away_penalties"), lit(0)))
    # Flag penalty shootout matches
    .withColumn("decided_by_penalties", when(col("penalty_shootout") == 1, lit(True)).otherwise(lit(False)))
    # Compute total goals per match
    .withColumn("total_goals", col("home_goals") + col("away_goals"))
    # Add silver layer processing timestamp
    .withColumn("_silver_processed_at", current_timestamp())
)

# Cache silver DataFrame — it's used by both validation (Cell 5) and write (Cell 6).
# This addresses the Spark advisory 'Accelerate queries with Delta' by avoiding
# recomputation of the transformation DAG across multiple downstream actions.
matches_silver_df.cache()

print("Silver transformation defined. Cached for downstream validation & write.")
display(matches_silver_df.limit(10))

In [0]:
from pyspark.sql.functions import (
    count, countDistinct, sum as spark_sum,
    min as spark_min, max as spark_max, col, when, lit
)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def validate_silver_quality(df: DataFrame, today: date) -> dict:
    """Run all silver-layer quality checks in a single-pass aggregation.

    Optimization: Consolidated multiple .filter().count() calls into one .agg() pass.
    This addresses the Spark advisory 'Accelerate queries with Delta' which flagged
    repeated highly-selective filter scans against source data.
    """

    critical_columns = ["match_id", "match_date", "home_team", "away_team", "home_goals", "away_goals", "winner"]

    # --- Single-pass aggregation for all metrics ---
    agg_exprs = [
        count("*").alias("total_rows"),
        countDistinct("match_id").alias("distinct_matches"),
        spark_min("match_date").alias("min_date"),
        spark_max("match_date").alias("max_date"),
        # Check for invalid goal values (negative)
        spark_sum(when(col("home_goals") < 0, 1).otherwise(0)).alias("negative_home_goals"),
        spark_sum(when(col("away_goals") < 0, 1).otherwise(0)).alias("negative_away_goals"),
        # Check for unreasonably high goals (> MAX_GOALS_PER_TEAM)
        spark_sum(when(col("home_goals") > MAX_GOALS_PER_TEAM, 1).otherwise(0)).alias("extreme_home_goals"),
        spark_sum(when(col("away_goals") > MAX_GOALS_PER_TEAM, 1).otherwise(0)).alias("extreme_away_goals"),
        # Check for future dates
        spark_sum(when(col("match_date") > lit(str(today)), 1).otherwise(0)).alias("future_dates"),
    ]
    # Add null counts for each critical column
    for c in critical_columns:
        agg_exprs.append(spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(f"null_{c}"))

    stats = df.agg(*agg_exprs).collect()[0]

    # --- Extract metrics ---
    total_rows = stats["total_rows"]
    duplicate_count = total_rows - stats["distinct_matches"]
    metrics = {"total_rows": total_rows, "duplicate_match_ids": duplicate_count}

    # --- Report results ---
    print(f"{'PASS' if duplicate_count == 0 else 'WARN'} | Duplicate match_id values: {duplicate_count} (will be deduplicated)")

    print("\n--- Null Check on Critical Fields ---")
    for c in critical_columns:
        null_count = int(stats[f"null_{c}"])
        null_pct = (null_count / total_rows * 100) if total_rows > 0 else 0
        status = "PASS" if null_pct == 0 else ("WARN" if null_pct < 5 else "FAIL")
        print(f"  {status} | {c}: {null_count} nulls ({null_pct:.1f}%)")
        metrics[f"null_{c}"] = null_count

    # --- Domain validation ---
    neg_goals = int(stats["negative_home_goals"]) + int(stats["negative_away_goals"])
    extreme_goals = int(stats["extreme_home_goals"]) + int(stats["extreme_away_goals"])
    future_dates = int(stats["future_dates"])

    print(f"\n{'PASS' if neg_goals == 0 else 'FAIL'} | Negative goal values: {neg_goals}")
    print(f"{'PASS' if extreme_goals == 0 else 'WARN'} | Extreme goals (>{MAX_GOALS_PER_TEAM}): {extreme_goals}")
    print(f"{'PASS' if future_dates == 0 else 'FAIL'} | Future date records: {future_dates}")
    metrics["negative_goals"] = neg_goals
    metrics["extreme_goals"] = extreme_goals
    metrics["future_dates"] = future_dates

    assert total_rows > 0, "FATAL: No rows in silver layer!"
    print(f"\nPASS | Total rows: {total_rows:,}")
    print(f"INFO | Date range: {stats['min_date']} to {stats['max_date']}")

    has_critical_issues = neg_goals > 0 or future_dates > 0
    print(f"\n{'=' * 50}")
    print(f"OVERALL: {'FAIL - Critical issues found' if has_critical_issues else 'PASS - All checks passed'}")
    return metrics


quality_metrics = validate_silver_quality(matches_silver_df, TODAY)

# --- Deduplication: Keep latest ingested record per match_id ---
# This ensures the Data Contract guarantee of unique match_id in silver output.
if quality_metrics["duplicate_match_ids"] > 0:
    window_spec = Window.partitionBy("match_id").orderBy(col("_ingestion_timestamp").desc())
    matches_silver_df = (
        matches_silver_df
        .withColumn("_row_num", row_number().over(window_spec))
        .filter(col("_row_num") == 1)
        .drop("_row_num")
    )
    matches_silver_df.cache()
    print(f"\nDeduplicated: {quality_metrics['total_rows']:,} -> {matches_silver_df.count():,} records")
else:
    print(f"\nNo duplicates — skipping deduplication step.")

In [0]:
# --- Schema/Database Creation (Unity Catalog ready) ---
if CATALOG != "hive_metastore":
    spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_NAME}")

# --- Write silver Delta table ---
# Optimization: Use quality_metrics from previous cell to avoid redundant scans.
# This addresses the Spark advisory 'Accelerate queries with Delta' which flagged
# repeated highly-selective filter scans against source data.
# Since validation passed, write directly without re-filtering.
(
    matches_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_FULL_TABLE)
)

# --- Add table metadata for discoverability ---
current_user = spark.sql("SELECT current_user()").collect()[0][0]

spark.sql(f"""
    ALTER TABLE {SILVER_FULL_TABLE} SET TBLPROPERTIES (
        'layer' = 'silver',
        'pipeline' = 'match_prediction',
        'source_table' = '{BRONZE_TABLE}',
        'quality_tier' = 'cleansed',
        'environment' = '{ENV}',
        'refresh_mode' = '{REFRESH_MODE}',
        'downstream_tables' = '{CATALOG}.{SCHEMA_NAME}.team_form_gold,{CATALOG}.{SCHEMA_NAME}.head_to_head_gold,{CATALOG}.{SCHEMA_NAME}.team_performance_gold',
        'transformations' = 'winner_derivation,penalty_resolution,string_standardization,deduplication'
    )
""")

spark.sql(f"COMMENT ON TABLE {SILVER_FULL_TABLE} IS 'Silver layer: Cleansed FIFA World Cup match data with derived winner (incl. penalty resolution), standardized strings, deduplication. Feeds 8 gold tables.'")

# --- Column-level comments for discoverability ---
silver_column_comments = {
    "match_id": "Unique match identifier (deduplicated)",
    "match_date": "Date the match was played (validated: 1930+, not future)",
    "home_team": "Home team name (trimmed, standardized)",
    "away_team": "Away team name (trimmed, standardized)",
    "home_goals": "Goals scored by home team (validated: 0-20 range)",
    "away_goals": "Goals scored by away team (validated: 0-20 range)",
    "winner": "Match winner (resolves penalty shootouts correctly)",
    "match_outcome": "Categorical outcome: home_win / away_win / draw",
    "goal_difference": "Absolute goal difference (home_goals - away_goals)",
    "total_goals": "Sum of both teams goals in the match",
    "decided_by_penalties": "Whether match winner was decided by penalty shootout",
    "tournament_name": "World Cup edition (trimmed)",
    "stage_name": "Tournament stage (trimmed, standardized)",
    "country_name": "Host country (trimmed)",
    "_silver_processed_at": "Timestamp when record was processed into silver",
}

for col_name, comment in silver_column_comments.items():
    try:
        spark.sql(f"ALTER TABLE {SILVER_FULL_TABLE} ALTER COLUMN `{col_name}` COMMENT '{comment}'")
    except Exception:
        pass  # Column may not exist

print(f"\n\u2705 Silver table registered: {SILVER_FULL_TABLE}")
print(f"   TBLPROPERTIES: layer, pipeline, source_table, quality_tier, owner, downstream, transformations")
print(f"   Column comments: {len(silver_column_comments)} columns documented")

# Verify
silver_verify_df = spark.table(SILVER_FULL_TABLE)
print(f"   Verified row count: {silver_verify_df.count():,}")
print(f"   Columns: {len(silver_verify_df.columns)}")

# Release cache — no longer needed after write
matches_silver_df.unpersist()

In [0]:
from pyspark.sql.functions import col, count, when, min as spark_min, max as spark_max

# ============================================================
# DATA CONTRACT: Silver layer output guarantees
# These assertions form the contract between silver → gold.
# Validates: completeness, correctness, consistency.
# ============================================================

silver_df = spark.table(SILVER_FULL_TABLE)

contract_checks = []

# 1. Row count matches bronze (no data loss during transformation)
bronze_count = spark.table(BRONZE_TABLE).count()
silver_count = silver_df.count()
assert silver_count >= bronze_count * 0.95, (
    f"CONTRACT VIOLATION: Silver ({silver_count}) has >5% fewer rows than bronze ({bronze_count})"
)
contract_checks.append(f"\u2705 Row preservation: {silver_count}/{bronze_count} ({silver_count/bronze_count*100:.1f}%)")

# 2. No null critical fields (guaranteed to downstream)
critical_cols = ["match_id", "match_date", "home_team", "away_team", "home_goals", "away_goals", "winner", "match_outcome"]
null_counts = silver_df.agg(
    *[count(when(col(c).isNull(), 1)).alias(c) for c in critical_cols]
).collect()[0]

for c in critical_cols:
    null_val = null_counts[c]
    assert null_val == 0, f"CONTRACT VIOLATION: {c} has {null_val} nulls"
contract_checks.append(f"\u2705 Null-free guarantee: All {len(critical_cols)} critical columns have zero nulls")

# 3. Winner derivation correctness (every non-draw must have a valid winner)
inconsistent = silver_df.filter(
    (col("match_outcome") != "draw") & 
    ((col("winner") == "Draw") | col("winner").isNull())
).count()
assert inconsistent == 0, f"CONTRACT VIOLATION: {inconsistent} non-draw matches have 'Draw' as winner"
contract_checks.append(f"\u2705 Winner logic: Consistent with match_outcome (incl. penalty resolution)")

# 4. Goal range validation (domain constraint)
out_of_range = silver_df.filter(
    (col("home_goals") < 0) | (col("home_goals") > MAX_GOALS_PER_TEAM) |
    (col("away_goals") < 0) | (col("away_goals") > MAX_GOALS_PER_TEAM)
).count()
assert out_of_range == 0, f"CONTRACT VIOLATION: {out_of_range} rows with goals outside 0-{MAX_GOALS_PER_TEAM}"
contract_checks.append(f"\u2705 Goal range: All values in [0, {MAX_GOALS_PER_TEAM}]")

# 5. No duplicate match_ids
duplicates = silver_df.groupBy("match_id").count().filter(col("count") > 1).count()
assert duplicates == 0, f"CONTRACT VIOLATION: {duplicates} duplicate match_id values in silver"
contract_checks.append(f"\u2705 Uniqueness: Zero duplicate match_ids")

# 6. Freshness metadata present
has_timestamp = silver_df.filter(col("_silver_processed_at").isNotNull()).count()
assert has_timestamp == silver_count, f"CONTRACT VIOLATION: {silver_count - has_timestamp} rows missing _silver_processed_at"
contract_checks.append(f"\u2705 Freshness metadata: All rows have _silver_processed_at")

print("\u2501" * 50)
print("SILVER DATA CONTRACT: ALL CHECKS PASSED")
print("\u2501" * 50)
for check in contract_checks:
    print(f"  {check}")
print(f"\n  Contract version: 1.0 | Enforced at: {TODAY}")

In [0]:
import json

# --- Exit status for orchestration (Lakeflow Jobs) ---
exit_status = {
    "status": "SUCCESS",
    "environment": ENV,
    "catalog": CATALOG,
    "silver_table": SILVER_FULL_TABLE,
    "source_table": BRONZE_TABLE,
    "rows_processed": quality_metrics["total_rows"],
    "duplicates_removed": quality_metrics["duplicate_match_ids"],
    "data_contract": "PASSED",
    "refresh_mode": REFRESH_MODE,
    "run_timestamp": datetime.now().isoformat(),
    "has_quality_warnings": quality_metrics["duplicate_match_ids"] > 0 or quality_metrics["negative_goals"] > 0,
    "downstream": [
        f"{CATALOG}.{SCHEMA_NAME}.team_form_gold",
        f"{CATALOG}.{SCHEMA_NAME}.head_to_head_gold",
        f"{CATALOG}.{SCHEMA_NAME}.team_performance_gold",
    ],
}

print("=" * 50)
print("SILVER LAYER PIPELINE COMPLETE")
print("=" * 50)
print(json.dumps(exit_status, indent=2))

# Return structured output for downstream tasks in Jobs
dbutils.notebook.exit(json.dumps(exit_status))